In [1]:
# Jupyter Notebook - 代码
# 导入必要的库
import matplotlib.pyplot as plt
import numpy as np
import os,shutil
import tensorflow as tf
import seaborn as sns
from tqdm import tqdm
import datetime
import tensorflow_model_optimization as tfmot
from lib import AU
# 设定日志级别
tf.get_logger().setLevel('ERROR')

# 🔹 超参数
IMG_SIZE = (160, 160)
AUTOTUNE = tf.data.AUTOTUNE
IMG_SHAPE = IMG_SIZE + (3,)

# 创建model目录（如果不存在）
model_dir = 'model'
os.makedirs(model_dir, exist_ok=True)

阶段一

In [2]:
# 检查缓存目录是否存在并删除
folder = 'cache'
if os.path.exists(folder) and os.path.isdir(folder):
    try:
        shutil.rmtree(folder)
        print(f"成功删除目录: {folder}")
    except Exception as e:
        print(f"删除失败，错误信息: {e}")
else:
    print(f"目录 '{folder}' 不存在")
    
BATCH_SIZE = 16

# 🔹 数据集路径
cache_dir = os.path.join('cache')
os.makedirs(cache_dir, exist_ok=True)

base_dir = '../Datasets/smartcar26_dataset'  # 替换为你的数据集路径
train_dir = os.path.join(base_dir, 'train')
valid_dir = os.path.join(base_dir, 'val')

train_dataset_raw = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, 
    batch_size=BATCH_SIZE, 
    image_size=IMG_SIZE)

validation_dataset_raw = tf.keras.preprocessing.image_dataset_from_directory(
    valid_dir, 
    batch_size=BATCH_SIZE, 
    image_size=IMG_SIZE)

class_names = train_dataset_raw.class_names
print("Class Names:", class_names)

# 使用磁盘缓存，减小内存压力
train_cache_path = os.path.join(cache_dir, 'train_cache1')
val_cache_path = os.path.join(cache_dir, 'val_cache1')

train_dataset = (train_dataset_raw
                 .map(AU.preprocess_image, num_parallel_calls=AUTOTUNE)
                 .cache(train_cache_path)
                 .shuffle(1000, reshuffle_each_iteration=True)
                 .prefetch(AUTOTUNE))

validation_dataset = (validation_dataset_raw
                      .map(AU.preprocess_image, num_parallel_calls=AUTOTUNE)
                      .cache(val_cache_path)
                      .prefetch(AUTOTUNE))

# 关键：训练前先完整遍历一次，确保cache文件写完整，避免partial cache warning
# 注意：必须完整循环，不能使用 break
print("开始预热缓存（stage1）...")
for _ in tqdm(train_dataset, desc="Caching train"):
    pass
for _ in tqdm(validation_dataset, desc="Caching val"):
    pass
print("缓存预热完成（stage1）")

目录 'cache' 不存在
Found 18329 files belonging to 10 classes.
Found 4701 files belonging to 10 classes.
Class Names: ['00mickey_mouse', '01pikachu', '02spongebob_squarepants', '03pleasant_sheep', '04donald_duck', '05nezha', '06big_head_son', '07gg_bond', '08calabash_brothers', '09grey_wolf']
开始预热缓存（stage1）...


Caching val: 100%|██████████| 294/294 [00:02<00:00, 137.33it/s]

缓存预热完成（stage1）


In [3]:
# 🔹 构建模型

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE, 
    include_top=False, 
    pooling = 'avg', 
    alpha=0.35, 
    # include_preprocessing=False,
    weights='imagenet')

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),
    base_model,
    tf.keras.layers.Dropout(0.8),
    tf.keras.layers.Dense(len(class_names),activation='softmax')
])
model.build((None, 160, 160, 3))
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 160, 160, 3)       0         
                                                                 
 mobilenetv2_0.35_160 (Funct  (None, 1280)             410208    
 ional)                                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 10)                12810     
                                                                 
Total params: 423,018
Trainable params: 408,938
Non-trainable params: 14,080
_________________________________________________________________


In [ ]:
# 编译模型

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.00001, decay_steps=len(train_dataset), decay_rate=0.99, staircase=True)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

history1 = model.fit(train_dataset,
                    validation_data=validation_dataset,  
                    epochs=50, 
                    callbacks=[early_stopping]
                    )   

# 先保存到临时路径，在后面确定了 model_folder_path 后再移动或复制
os.makedirs('./path', exist_ok=True)
model.save('./path/stage1_model.h5')

Epoch 1/50
1146/1146 [==============================] - 29s 18ms/step - loss: 2.6976 - accuracy: 0.2217 - val_loss: 1.0145 - val_accuracy: 0.7635
Epoch 2/50
1146/1146 [==============================] - 27s 17ms/step - loss: 1.4698 - accuracy: 0.5131 - val_loss: 0.3625 - val_accuracy: 0.9313
Epoch 3/50
1146/1146 [==============================] - 26s 17ms/step - loss: 0.8025 - accuracy: 0.7352 - val_loss: 0.1677 - val_accuracy: 0.9638
Epoch 4/50
1146/1146 [==============================] - 26s 17ms/step - loss: 0.4974 - accuracy: 0.8398 - val_loss: 0.0991 - val_accuracy: 0.9743
Epoch 5/50
1146/1146 [==============================] - 26s 17ms/step - loss: 0.3381 - accuracy: 0.8919 - val_loss: 0.0630 - val_accuracy: 0.9843
Epoch 6/50
1146/1146 [==============================] - 24s 17ms/step - loss: 0.2443 - accuracy: 0.9223 - val_loss: 0.0434 - val_accuracy: 0.9885
Epoch 7/50
1146/1146 [==============================] - 25s 18ms/step - loss: 0.1865 - accuracy: 0.9419 - val_loss: 0.0314 -

阶段二

In [ ]:
model = tf.keras.models.load_model('./path/stage1_model.h5')
model.summary()

In [ ]:
# 检查缓存目录是否存在并删除
folder = 'cache'
if os.path.exists(folder) and os.path.isdir(folder):
    try:
        shutil.rmtree(folder)
        print(f"成功删除目录: {folder}")
    except Exception as e:
        print(f"删除失败，错误信息: {e}")
else:
    print(f"目录 '{folder}' 不存在")

BATCH_SIZE = 16

# 🔹 数据集路径
cache_dir = os.path.join('cache')
os.makedirs(cache_dir, exist_ok=True)

base_dir = '../Datasets/smartcar26_dataset'  # 替换为你的数据集路径
train_dir = os.path.join(base_dir, 'train')
valid_dir = os.path.join(base_dir, 'val')

train_dataset_raw = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, 
    batch_size=BATCH_SIZE, 
    image_size=IMG_SIZE)

validation_dataset_raw = tf.keras.preprocessing.image_dataset_from_directory(
    valid_dir, 
    batch_size=BATCH_SIZE, 
    image_size=IMG_SIZE)

class_names = train_dataset_raw.class_names
print("Class Names:", class_names)

train_cache_path = os.path.join(cache_dir, 'train_cache2')
val_cache_path = os.path.join(cache_dir, 'val_cache2')

train_dataset = (train_dataset_raw
                 .map(AU.preprocess_image_aug, num_parallel_calls=AUTOTUNE)
                 .cache(train_cache_path)
                 .shuffle(1000, reshuffle_each_iteration=True)
                 .prefetch(AUTOTUNE))

validation_dataset = (validation_dataset_raw
                      .map(AU.preprocess_image, num_parallel_calls=AUTOTUNE)
                      .cache(val_cache_path)
                      .prefetch(AUTOTUNE))

print("开始预热缓存（stage2）...")
for _ in tqdm(train_dataset, desc="Caching train stage2"):
    pass
for _ in tqdm(validation_dataset, desc="Caching val stage2"):
    pass
print("缓存预热完成（stage2）")

In [ ]:
# 编译模型
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.00001, decay_steps=len(train_dataset), decay_rate=0.99, staircase=True)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

# 训练第二阶段
early_stopping = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

history2 = model.fit(train_dataset,
                    validation_data=validation_dataset,
                    epochs=5, 
                    callbacks=[early_stopping]
                    )

# 保存二阶段模型
model.save('./path/stage2_model.h5')

In [ ]:
# 🔹 直接导出为TFLite格式 (无需保存H5)
def representative_dataset():
    # 单独构建校准数据管道，避免对已cache的数据集再take导致部分缓存被丢弃
    calibration_dataset = (validation_dataset_raw
                           .map(AU.preprocess_image, num_parallel_calls=AUTOTUNE)
                           .take(500)
                           .cache()
                           .prefetch(AUTOTUNE))

    for images, _ in tqdm(calibration_dataset, desc="Calibration"):
        yield [tf.cast(images, tf.float32)]  # 输入需为浮点型

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8   # 输入为uint8 (0-255) 
converter.inference_output_type = tf.float32  # 输出为float32，解决置信度精度问题

tflite_model = converter.convert()


# 保存带时间戳的TFLite模型和标签
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
model_folder_name = f"model_{timestamp}"
model_folder_path = f"./model/{model_folder_name}"
os.makedirs(model_folder_path, exist_ok=True)

# 保存模型文件
output_path = os.path.join(model_folder_path, f"{model_folder_name}.tflite")
with open(output_path, 'wb') as f:
    f.write(tflite_model)

# 保存标签文件
labels_path = os.path.join(model_folder_path, "labels.txt")
with open(labels_path, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")

# 将中间阶段模型 stage1_model.h5 和 stage2_model.h5 也保存到当前文件夹
for stage_name in ["stage1_model.h5", "stage2_model.h5"]:
    stage_path = os.path.join('./path', stage_name)
    if os.path.exists(stage_path):
        shutil.copy(stage_path, os.path.join(model_folder_path, stage_name))
        print(f"模型 {stage_name} 已保存至: {model_folder_path}")

print(f"TFLite模型和标签已保存至: {model_folder_path}")

target_dir = "./model" 
# 直接匹配当前目录下的 .h5 文件
for file in os.listdir(target_dir): 
    if file.endswith(".h5"):
        file_path = os.path.join(target_dir, file)
        try:
            os.remove(file_path)
            # print(f"已删除文件: {file_path}")
        except Exception as e:
            print(f"删除 {file_path} 时出错: {e}")

In [ ]:
from sklearn.metrics import confusion_matrix
# 混淆矩阵
y_pred = np.argmax(model.predict(validation_dataset), axis=1)
y_true = np.concatenate([labels.numpy() for _, labels in validation_dataset])

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap="Blues", fmt="d", 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

# 保存混淆矩阵到模型文件夹
if 'model_folder_path' in locals():
    cm_path = os.path.join(model_folder_path, "confusion_matrix.png")
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    print(f"混淆矩阵已保存至: {cm_path}")

plt.show()

In [ ]:
from lib import polt_improved

stage_names = ["history"]
history_list = [history1,history2]

# 绘图并保存到模型文件夹
if 'model_folder_path' in locals():
    polt_improved.plot_combined_curves_improved(history_list, save_dir=model_folder_path)
    print(f"训练曲线已保存至: {model_folder_path}/training_curves.png")
else:
    polt_improved.plot_combined_curves_improved(history_list)

# 检查缓存目录是否存在并删除
folder = 'cache'
if os.path.exists(folder) and os.path.isdir(folder):
    try:
        shutil.rmtree(folder)
        print(f"成功删除目录: {folder}")
    except Exception as e:
        print(f"删除失败，错误信息: {e}")
else:
    print(f"目录 '{folder}' 不存在")

In [ ]:
# 🔹 调用 model_test.py 进行模型测试
import model_test

if 'output_path' in locals() and os.path.exists(output_path):
    # 确定测试集路径
    test_dir = os.path.join(base_dir, 'test')
    if not os.path.exists(test_dir):
        # 尝试备选路径
        test_dir = os.path.join(os.path.dirname(base_dir), 'smartcar26_dataset', 'test')
    
    if os.path.exists(test_dir):
        print(f"调用 model_test 测试模型: {output_path}")
        model_test.main(model_path=output_path, test_dir=test_dir)
    else:
        print(f"未找到测试集目录，请检查路径。")
else:
    print("未找到导出的模型，请先运行导出单元格。")